In [ ]:
# Method 6: Graph-Based Entity Retrieval on MuSiQue
# Complete single-cell setup

from google.colab import drive
drive.mount('/content/drive')

!pip -q install transformers accelerate sentencepiece rank_bm25 spacy
!python -m spacy download en_core_web_sm -q

import re, json, time, torch
from pathlib import Path
from collections import Counter, defaultdict
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import spacy

PROJECT_DIR = Path("/content/drive/MyDrive/421-Project")
device = "cuda"

# Load MuSiQue dev set
dev_data = []
with open(PROJECT_DIR / "musique_ans_v1.0_dev.jsonl") as f:
    for line in f:
        dev_data.append(json.loads(line))
print(f"Dev: {len(dev_data)}")

# Helpers
def normalize_text(s):
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9\s]", "", s)
    return " ".join(s.split())

def compute_em(pred, gold):
    return int(normalize_text(pred) == normalize_text(gold))

def compute_f1(pred, gold):
    pt, gt = normalize_text(pred).split(), normalize_text(gold).split()
    common = Counter(pt) & Counter(gt)
    ns = sum(common.values())
    if not pt or not gt: return int(pt == gt)
    if ns == 0: return 0.0
    p, r = ns/len(pt), ns/len(gt)
    return 2*p*r/(p+r)

def compute_em_with_aliases(pred, example):
    candidates = [example["answer"]] + example.get("answer_aliases", [])
    return max(compute_em(pred, g) for g in candidates)

def compute_f1_with_aliases(pred, example):
    candidates = [example["answer"]] + example.get("answer_aliases", [])
    return max(compute_f1(pred, g) for g in candidates)

def simple_tokenize(text):
    return re.findall(r"\w+", text.lower())

def get_hop(ex):
    eid = ex["id"]
    if eid.startswith("2hop"): return "2-hop"
    if eid.startswith("3hop"): return "3-hop"
    if eid.startswith("4hop"): return "4-hop"
    return "other"

def paragraph_to_text(p):
    return f"Title: {p['title']}\n{p['paragraph_text']}"

def build_supporting_context(ex):
    return "\n\n".join(paragraph_to_text(p) for p in ex["paragraphs"] if p["is_supporting"])

def build_bm25_topk_context(ex, k=5):
    paras = ex["paragraphs"]
    texts = [paragraph_to_text(p) for p in paras]
    bm25 = BM25Okapi([simple_tokenize(t) for t in texts])
    scores = bm25.get_scores(simple_tokenize(ex["question"]))
    ranked = sorted(zip(paras, texts, scores), key=lambda x: x[2], reverse=True)[:k]
    ctx = "\n\n".join(t for _, t, _ in ranked)
    supp = set(p["title"] for p in paras if p["is_supporting"])
    ret = set(p["title"] for p, _, _ in ranked)
    recall = len(supp & ret) / len(supp) if supp else 0
    return ctx, recall, int(supp <= ret)

def build_qa_prompt(question, context):
    return f"Answer the question using the context below.\n\nQuestion: {question}\n\nContext:\n{context}\n\nAnswer:"

# Load spaCy
nlp = spacy.load("en_core_web_sm")
print("spaCy loaded.")

# Graph retrieval — adapted for MuSiQue paragraph format
def extract_entities(text):
    doc = nlp(text[:5000])
    entities = set()
    for ent in doc.ents:
        entities.add(ent.text.lower().strip())
    for chunk in doc.noun_chunks:
        if len(chunk.text.split()) <= 4:
            entities.add(chunk.text.lower().strip())
    return entities

def hybrid_graph_retrieve_musique(ex, k=2):
    """
    Hybrid BM25 + entity graph retrieval for MuSiQue.
    MuSiQue paragraphs have 'title' and 'paragraph_text' fields.
    """
    question = ex["question"]
    paras = ex["paragraphs"]
    para_texts = [paragraph_to_text(p) for p in paras]

    # BM25 for hop-1
    bm25 = BM25Okapi([simple_tokenize(t) for t in para_texts])
    bm25_scores = bm25.get_scores(simple_tokenize(question))
    bm25_ranked = sorted(range(len(bm25_scores)),
                         key=lambda i: bm25_scores[i], reverse=True)

    best_pair, best_pair_score = None, -1

    for hop1_idx in bm25_ranked[:5]:
        hop1_title = paras[hop1_idx]["title"]
        hop1_text = paras[hop1_idx]["paragraph_text"].lower()
        hop1_full = (hop1_title + " " + hop1_text).lower()
        hop1_entities = extract_entities(hop1_full)

        for j, p in enumerate(paras):
            if j == hop1_idx: continue
            other_title = p["title"]
            other_text = p["paragraph_text"].lower()
            other_full = (other_title + " " + other_text).lower()
            other_title_lower = other_title.lower()

            connection_score = 0

            # Strong: hop2 title appears in hop1 text
            if other_title_lower in hop1_full:
                connection_score += 50
            # Strong: hop1 title appears in hop2 text
            if hop1_title.lower() in other_full:
                connection_score += 50
            # Medium: shared named entities
            other_entities = extract_entities(other_full)
            shared = {e for e in (hop1_entities & other_entities) if len(e) > 3}
            connection_score += len(shared) * 2
            # Bonus: hop2 has question relevance
            q_overlap = len(extract_entities(question) & other_entities)
            connection_score += q_overlap * 3

            if connection_score > 0:
                total = bm25_scores[hop1_idx] + connection_score
                if total > best_pair_score:
                    best_pair_score = total
                    best_pair = (hop1_idx, j)

    selected = list(best_pair) if best_pair and best_pair_score > 0 else bm25_ranked[:k]

    context = "\n\n".join(paragraph_to_text(paras[i]) for i in selected[:k])
    supp = set(p["title"] for p in paras if p["is_supporting"])
    ret = set(paras[i]["title"] for i in selected[:k])
    recall = len(supp & ret) / len(supp) if supp else 0
    full_recall = int(supp <= ret)

    return context, recall, full_recall, selected[:k]

print("Graph retrieval ready.")

# Load XXL
MODEL_NAME = "google/flan-t5-xxl"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16,
    device_map="auto", low_cpu_mem_usage=True
)

def run_model(prompt, max_tokens=32):
    inputs = tokenizer(prompt, return_tensors="pt",
                      truncation=True, max_length=4096).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_tokens,
                                  num_beams=4, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

print(f"XXL loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print("Ready.")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 81.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Dev: 2417
spaCy loaded.
Graph retrieval ready.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/674 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/559 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

XXL loaded: 0.0 GB
Ready.


In [ ]:
import torch
import gc

# Clear any cached memory first
gc.collect()
torch.cuda.empty_cache()

# Memory stats
allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved() / 1e9
total     = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU:        {torch.cuda.get_device_name(0)}")
print(f"Total VRAM: {total:.1f} GB")
print(f"Allocated:  {allocated:.1f} GB")
print(f"Reserved:   {reserved:.1f} GB")
print(f"Free:       {total - reserved:.1f} GB")
print(f"Fragmentation: {reserved - allocated:.1f} GB")

GPU:        NVIDIA A100-SXM4-40GB
Total VRAM: 42.4 GB
Allocated:  0.0 GB
Reserved:   0.0 GB
Free:       42.4 GB
Fragmentation: 0.0 GB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install rank_bm25 spacy
!python -m spacy download en_core_web_sm -q

import re, json, time, torch
from pathlib import Path
from collections import Counter, defaultdict
from rank_bm25 import BM25Okapi
import spacy

PROJECT_DIR = Path("/content/drive/MyDrive/421-Project")
device = "cuda"

dev_data = []
with open(PROJECT_DIR / "musique_ans_v1.0_dev.jsonl") as f:
    for line in f:
        dev_data.append(json.loads(line))

nlp = spacy.load("en_core_web_sm")
print(f"Dev: {len(dev_data)}")
print(f"spaCy loaded.")
print(f"GPU allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 125.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Dev: 2417
spaCy loaded.
GPU allocated: 0.0 GB


In [ ]:
def normalize_text(s):
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9\s]", "", s)
    return " ".join(s.split())

def compute_em(pred, gold):
    return int(normalize_text(pred) == normalize_text(gold))

def compute_f1(pred, gold):
    pt, gt = normalize_text(pred).split(), normalize_text(gold).split()
    common = Counter(pt) & Counter(gt)
    ns = sum(common.values())
    if not pt or not gt: return int(pt == gt)
    if ns == 0: return 0.0
    p, r = ns/len(pt), ns/len(gt)
    return 2*p*r/(p+r)

def compute_em_with_aliases(pred, example):
    candidates = [example["answer"]] + example.get("answer_aliases", [])
    return max(compute_em(pred, g) for g in candidates)

def compute_f1_with_aliases(pred, example):
    candidates = [example["answer"]] + example.get("answer_aliases", [])
    return max(compute_f1(pred, g) for g in candidates)

def simple_tokenize(text):
    return re.findall(r"\w+", text.lower())

def get_hop(ex):
    eid = ex["id"]
    if eid.startswith("2hop"): return "2-hop"
    if eid.startswith("3hop"): return "3-hop"
    if eid.startswith("4hop"): return "4-hop"
    return "other"

def paragraph_to_text(p):
    return f"Title: {p['title']}\n{p['paragraph_text']}"

def build_bm25_topk_context(ex, k=2):
    paras = ex["paragraphs"]
    texts = [paragraph_to_text(p) for p in paras]
    bm25 = BM25Okapi([simple_tokenize(t) for t in texts])
    scores = bm25.get_scores(simple_tokenize(ex["question"]))
    ranked = sorted(zip(paras, texts, scores), key=lambda x: x[2], reverse=True)[:k]
    ctx = "\n\n".join(t for _, t, _ in ranked)
    supp = set(p["title"] for p in paras if p["is_supporting"])
    ret = set(p["title"] for p, _, _ in ranked)
    recall = len(supp & ret) / len(supp) if supp else 0
    return ctx, recall, int(supp <= ret)

def build_qa_prompt(question, context):
    return f"Answer the question using the context below.\n\nQuestion: {question}\n\nContext:\n{context}\n\nAnswer:"

def extract_entities(text):
    doc = nlp(text[:5000])
    entities = set()
    for ent in doc.ents:
        entities.add(ent.text.lower().strip())
    for chunk in doc.noun_chunks:
        if len(chunk.text.split()) <= 4:
            entities.add(chunk.text.lower().strip())
    return entities

def hybrid_graph_retrieve_musique(ex, k=2):
    question = ex["question"]
    paras = ex["paragraphs"]
    para_texts = [paragraph_to_text(p) for p in paras]

    bm25 = BM25Okapi([simple_tokenize(t) for t in para_texts])
    bm25_scores = bm25.get_scores(simple_tokenize(question))
    bm25_ranked = sorted(range(len(bm25_scores)),
                         key=lambda i: bm25_scores[i], reverse=True)

    best_pair, best_pair_score = None, -1

    for hop1_idx in bm25_ranked[:5]:
        hop1_title = paras[hop1_idx]["title"]
        hop1_text = paras[hop1_idx]["paragraph_text"].lower()
        hop1_full = (hop1_title + " " + hop1_text).lower()
        hop1_entities = extract_entities(hop1_full)

        for j, p in enumerate(paras):
            if j == hop1_idx: continue
            other_title = p["title"]
            other_text = p["paragraph_text"].lower()
            other_full = (other_title + " " + other_text).lower()

            connection_score = 0
            if other_title.lower() in hop1_full: connection_score += 50
            if hop1_title.lower() in other_full: connection_score += 50
            other_entities = extract_entities(other_full)
            shared = {e for e in (hop1_entities & other_entities) if len(e) > 3}
            connection_score += len(shared) * 2
            q_overlap = len(extract_entities(question) & other_entities)
            connection_score += q_overlap * 3

            if connection_score > 0:
                total = bm25_scores[hop1_idx] + connection_score
                if total > best_pair_score:
                    best_pair_score = total
                    best_pair = (hop1_idx, j)

    selected = list(best_pair) if best_pair and best_pair_score > 0 else bm25_ranked[:k]
    context = "\n\n".join(paragraph_to_text(paras[i]) for i in selected[:k])
    supp = set(p["title"] for p in paras if p["is_supporting"])
    ret = set(paras[i]["title"] for i in selected[:k])
    recall = len(supp & ret) / len(supp) if supp else 0
    return context, recall, int(supp <= ret), selected[:k]

print("All helpers and graph retrieval defined.")

All helpers and graph retrieval defined.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-xxl"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16,
    device_map="auto", low_cpu_mem_usage=True
)

def run_model(prompt, max_tokens=32):
    inputs = tokenizer(prompt, return_tensors="pt",
                      truncation=True, max_length=4096).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_tokens,
                                  num_beams=4, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

print(f"XXL loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print("Ready.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/674 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/559 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

XXL loaded: 26.6 GB
Ready.


In [ ]:
# Method 6: MuSiQue — Graph vs BM25 (k=2) full evaluation

total = len(dev_data)

em_graph = defaultdict(list)
f1_graph = defaultdict(list)
em_bm25  = defaultdict(list)
f1_bm25  = defaultdict(list)
graph_full_recall = 0
bm25_full_recall  = 0

start = time.time()

for i, ex in enumerate(dev_data, start=1):
    hop = get_hop(ex)
    q   = ex["question"]

    # Graph retrieval + XXL
    g_ctx, _, g_full, _ = hybrid_graph_retrieve_musique(ex, k=2)
    g_pred = run_model(build_qa_prompt(q, g_ctx))
    em_graph[hop].append(compute_em_with_aliases(g_pred, ex))
    f1_graph[hop].append(compute_f1_with_aliases(g_pred, ex))
    graph_full_recall += g_full

    # BM25 top-2 + XXL (same k=2 for fair comparison)
    b_ctx, _, b_full = build_bm25_topk_context(ex, k=2)
    b_pred = run_model(build_qa_prompt(q, b_ctx))
    em_bm25[hop].append(compute_em_with_aliases(b_pred, ex))
    f1_bm25[hop].append(compute_f1_with_aliases(b_pred, ex))
    bm25_full_recall += b_full

    if i % 300 == 0:
        elapsed = time.time() - start
        all_g = [s for h in em_graph for s in em_graph[h]]
        all_b = [s for h in em_bm25 for s in em_bm25[h]]
        print(f"[{i}/{total}] Graph EM={sum(all_g)/len(all_g):.3f} "
              f"BM25 EM={sum(all_b)/len(all_b):.3f} "
              f"G_recall={graph_full_recall/i:.3f} "
              f"B_recall={bm25_full_recall/i:.3f} "
              f"Time={elapsed:.0f}s")

print("\n" + "=" * 90)
print(f"{'Method':<25} {'2-hop':>10} {'3-hop':>10} {'4-hop':>10} "
      f"{'All EM':>10} {'All F1':>10} {'Full Rec.':>10}")
print("=" * 90)

for name, em_d, f1_d, fr in [
    ("M6: Graph + XXL", em_graph, f1_graph, graph_full_recall),
    ("BM25 top-2 + XXL", em_bm25,  f1_bm25,  bm25_full_recall),
]:
    all_em = [s for h in em_d for s in em_d[h]]
    all_f1 = [s for h in f1_d for s in f1_d[h]]
    print(f"{name:<25} "
          f"{sum(em_d['2-hop'])/len(em_d['2-hop']):>10.3f} "
          f"{sum(em_d['3-hop'])/len(em_d['3-hop']):>10.3f} "
          f"{sum(em_d['4-hop'])/len(em_d['4-hop']):>10.3f} "
          f"{sum(all_em)/len(all_em):>10.3f} "
          f"{sum(all_f1)/len(all_f1):>10.3f} "
          f"{fr/total:>10.3f}")

print("=" * 90)
print(f"\nComparison — B2 BM25 top-5 XXL: EM=0.167, F1=0.270")
print(f"Comparison — B3 Oracle XXL:     EM=0.491, F1=0.649")

[300/2417] Graph EM=0.177 BM25 EM=0.130 G_recall=0.113 B_recall=0.067 Time=892s
[600/2417] Graph EM=0.178 BM25 EM=0.117 G_recall=0.132 B_recall=0.067 Time=1782s
[900/2417] Graph EM=0.173 BM25 EM=0.112 G_recall=0.141 B_recall=0.070 Time=2667s
[1200/2417] Graph EM=0.176 BM25 EM=0.109 G_recall=0.146 B_recall=0.073 Time=3565s
[1500/2417] Graph EM=0.176 BM25 EM=0.110 G_recall=0.146 B_recall=0.074 Time=4471s
[1800/2417] Graph EM=0.172 BM25 EM=0.106 G_recall=0.144 B_recall=0.071 Time=5368s
[2100/2417] Graph EM=0.176 BM25 EM=0.105 G_recall=0.146 B_recall=0.072 Time=6268s
[2400/2417] Graph EM=0.172 BM25 EM=0.112 G_recall=0.136 B_recall=0.070 Time=7210s

Method                         2-hop      3-hop      4-hop     All EM     All F1  Full Rec.
M6: Graph + XXL                0.219      0.113      0.133      0.171      0.264      0.135
BM25 top-2 + XXL               0.125      0.103      0.086      0.112      0.200      0.070

Comparison — B2 BM25 top-5 XXL: EM=0.167, F1=0.270
Comparison — B3 Ora